<!--
SPDX-License-Identifier: BUSL-1.1
Copyright 2026 Ryan Gillespie / Optitransfer

Licensed under the Business Source License 1.1 (the "License");
you may not use this file except in compliance with the License.
See https://github.com/mgillr/crdt-merge/blob/main/LICENSE
-->

# crdt-merge v0.8.1 — CRDT Architecture Benchmark

**Timestamp:** 2026-03-29T20:24:50.736939+00:00  
**Python:** 3.12.13  
**NumPy:** 2.4.4  
**Total Strategies:** 25  

This notebook presents the results of the comprehensive v0.8.1 CRDT architecture benchmark, testing all 25 merge strategies for CRDT law compliance, performance, integration, OR-Set semantics, memory estimation, and edge cases.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np

# Load benchmark results
with open('../benchmarks/v081/benchmark_results.json') as f:
    results = json.load(f)

print(f"crdt-merge v{results['version']} Benchmark Results")
print(f"Timestamp: {results['timestamp']}")
print(f"Python: {results['python_version']}")
print(f"Strategies: {results['total_strategies']}")

## 1. Strategy Registry Verification

All 25 registered strategies were successfully instantiated via `get_strategy()`.

| # | Strategy Name | Status |
|---|---|---|
| 1 | `ada_merging` | ✅ |
| 2 | `adarank` | ✅ |
| 3 | `dam` | ✅ |
| 4 | `dare` | ✅ |
| 5 | `dare_ties` | ✅ |
| 6 | `della` | ✅ |
| 7 | `emr` | ✅ |
| 8 | `evolutionary_merge` | ✅ |
| 9 | `fisher_merge` | ✅ |
| 10 | `genetic_merge` | ✅ |
| 11 | `led_merge` | ✅ |
| 12 | `linear` | ✅ |
| 13 | `model_breadcrumbs` | ✅ |
| 14 | `negative_merge` | ✅ |
| 15 | `regression_mean` | ✅ |
| 16 | `representation_surgery` | ✅ |
| 17 | `safe_merge` | ✅ |
| 18 | `slerp` | ✅ |
| 19 | `split_unlearn_merge` | ✅ |
| 20 | `star` | ✅ |
| 21 | `svd_knot_tying` | ✅ |
| 22 | `task_arithmetic` | ✅ |
| 23 | `ties` | ✅ |
| 24 | `weight_average` | ✅ |
| 25 | `weight_scope_alignment` | ✅ |


## 2. CRDT Compliance Matrix (25 strategies × 3 laws)

The two-layer CRDT architecture (set-union state + deterministic resolve) ensures all 25 strategies satisfy commutativity, associativity, and idempotency — verified with 10 randomized trials each.

| Strategy | Commutativity (State/Resolve) | Associativity (State/Resolve) | Idempotency (State/Resolve) |
|---|---|---|---|
| `ada_merging` | ✅ | ✅ | ✅ |
| `adarank` | ✅ | ✅ | ✅ |
| `dam` | ✅ | ✅ | ✅ |
| `dare` | ✅ | ✅ | ✅ |
| `dare_ties` | ✅ | ✅ | ✅ |
| `della` | ✅ | ✅ | ✅ |
| `emr` | ✅ | ✅ | ✅ |
| `evolutionary_merge` | ✅ | ✅ | ✅ |
| `fisher_merge` | ✅ | ✅ | ✅ |
| `genetic_merge` | ✅ | ✅ | ✅ |
| `led_merge` | ✅ | ✅ | ✅ |
| `linear` | ✅ | ✅ | ✅ |
| `model_breadcrumbs` | ✅ | ✅ | ✅ |
| `negative_merge` | ✅ | ✅ | ✅ |
| `regression_mean` | ✅ | ✅ | ✅ |
| `representation_surgery` | ✅ | ✅ | ✅ |
| `safe_merge` | ✅ | ✅ | ✅ |
| `slerp` | ✅ | ✅ | ✅ |
| `split_unlearn_merge` | ✅ | ✅ | ✅ |
| `star` | ✅ | ✅ | ✅ |
| `svd_knot_tying` | ✅ | ✅ | ✅ |
| `task_arithmetic` | ✅ | ✅ | ✅ |
| `ties` | ✅ | ✅ | ✅ |
| `weight_average` | ✅ | ✅ | ✅ |
| `weight_scope_alignment` | ✅ | ✅ | ✅ |

**CRDT Compliance: 25/25 strategies pass all 3 laws**

In [ ]:
# CRDT Compliance Heatmap
crdt = results['crdt_law_results']
strategies = sorted(crdt.keys())
laws = ['commutativity', 'associativity', 'idempotency']

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    matrix = []
    for s in strategies:
        row = []
        for law in laws:
            ok = crdt[s][law]['state'] and crdt[s][law]['resolve']
            row.append(1 if ok else 0)
        matrix.append(row)

    fig, ax = plt.subplots(figsize=(8, 12))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(laws)))
    ax.set_xticklabels(['Commutative', 'Associative', 'Idempotent'], fontsize=11)
    ax.set_yticks(range(len(strategies)))
    ax.set_yticklabels(strategies, fontsize=9)
    ax.set_title('CRDT Law Compliance Matrix (25 strategies)', fontsize=14)
    for i in range(len(strategies)):
        for j in range(len(laws)):
            ax.text(j, i, '✓' if matrix[i][j] else '✗', ha='center', va='center', fontsize=12)
    plt.tight_layout()
    plt.savefig('../benchmarks/v081/crdt_compliance_matrix.png', dpi=150)
    plt.show()
    print('Chart saved to benchmarks/v081/crdt_compliance_matrix.png')
except ImportError:
    print('matplotlib not available — printing text table')
    header = f"{'Strategy':<30s} {'Comm':>6s} {'Assoc':>6s} {'Idemp':>6s}"
    print(header)
    print('-' * len(header))
    for s in strategies:
        c = '✓' if crdt[s]['commutativity']['state'] else '✗'
        a = '✓' if crdt[s]['associativity']['state'] else '✗'
        i = '✓' if crdt[s]['idempotency']['state'] else '✗'
        print(f'{s:<30s} {c:>6s} {a:>6s} {i:>6s}')

## 3. Performance Benchmarks

Performance across three tensor sizes (50, 500, 5000) for all 25 strategies.

| Benchmark | Mean (ms) | Std (ms) |
|---|---|---|
| add_single | 21.231 | 6.905 |
| add_batch_100 | 1.936 | 1.727 |
| merge_pairwise | 0.038 | 0.022 |
| merge_many_10 | 0.018 | 0.004 |
| resolve_10_contributions | 343.830 | 1883.262 |
| serialization_roundtrip | 0.238 | 0.258 |


In [ ]:
# Performance Bar Chart
perf = results['performance']
labels = ['add_single', 'add_batch_100', 'merge_pairwise', 'merge_many_10',
          'resolve_10_contributions', 'serialization_roundtrip']
means = [perf[k]['mean_ms'] for k in labels]
stds = [perf[k]['std_ms'] for k in labels]

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color='steelblue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace('_', '\n') for l in labels], fontsize=9)
    ax.set_ylabel('Time (ms)')
    ax.set_title('crdt-merge v0.8.1 — Operation Performance (mean across all strategies & tensor sizes)')
    ax.set_yscale('log')
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{m:.2f}',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig('../benchmarks/v081/performance_chart.png', dpi=150)
    plt.show()
    print('Chart saved to benchmarks/v081/performance_chart.png')
except ImportError:
    print('matplotlib not available — text summary:')
    for l, m, s in zip(labels, means, stds):
        print(f'  {l:<35s} {m:>10.3f} ms  (±{s:.3f})')

## 4. Integration Tests

Results from the `ModelMerge.crdt_merge()` API, OR-Set semantics, memory estimation, and edge cases.

| Test | Passed | Details |
|---|---|---|
| crdt_merge API | ✅ | crdt_guaranteed=True, time=4.75 ms |
| OR-Set Semantics | ✅ | add-remove-readd, concurrent adds, merge after remove |
| Memory Estimation | ✅ | Validated for 1, 10, 50 contributions |
| Edge Cases | ✅ | Empty merge, single resolve, duplicate IDs, serialization roundtrip |


## Summary

| Metric | Value |
|---|---|
| **Total Tests** | 81 |
| **Passed** | 81 |
| **Failed** | 0 |
| **CRDT Compliance** | 25/25 |

### ✅ ALL TESTS PASSED — Ready for PyPI deployment

The two-layer CRDT architecture (set-union state + deterministic resolve) guarantees commutativity, associativity, and idempotency for **all 25 strategies**. This is the mathematical foundation that makes `crdt-merge` safe for fully decentralised model merging.

In [ ]:
# Final Summary
s = results['summary']
print('=' * 60)
print(f"crdt-merge v{results['version']} Benchmark Summary")
print('=' * 60)
print(f"  Total Tests:     {s['total_tests']}")
print(f"  Passed:          {s['passed']}")
print(f"  Failed:          {s['failed']}")
print(f"  CRDT Compliance: {s['crdt_compliance']}")
print('=' * 60)
if s['failed'] == 0:
    print('\n✅ ALL TESTS PASSED — v0.8.1 is ready for PyPI deployment')
else:
    print(f'\n⚠️ {s["failed"]} test(s) failed — review before deployment')